# 00 — Sweep driver

Runs the quality gate (`sweep --check`) on this machine — meant for a **Colab GPU kernel**
(open from VSCode connected to the kernel, or in Colab directly).

- Tier `S` — nano smoke run, CPU-friendly (CI parity)
- Tier `M` — the "actually learns" bar; EvalSpec thresholds gate here (GPU)
- Tier `L` — full budgets (GPU)

The report lands in `runs/sweep/<timestamp>/report.{md,json}` and renders below.

In [ ]:
import os, pathlib, subprocess, sys

if not pathlib.Path('main.py').exists():
    if not pathlib.Path('mini_networks').exists():
        subprocess.run(['git', 'clone', 'https://github.com/juan-garassino/mini_networks.git'], check=True)
    os.chdir('mini_networks')
    subprocess.run(['git', 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync'], check=True)
print('ready:', pathlib.Path.cwd())

In [ ]:
import torch

TIER = 'M'                 # S | M | L
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS = 'all'             # or comma-separated, e.g. 'gan,diffusion'
COMPOSITIONS = 'all'

print(f'tier={TIER} device={DEVICE}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'main.py', 'sweep', '--check',
    '--training_tier', TIER, '--device', DEVICE,
    '--models', MODELS, '--compositions', COMPOSITIONS,
]
proc = subprocess.run(cmd, text=True)
print('exit code:', proc.returncode)

In [ ]:
# Render the latest report inline
import pathlib
from IPython.display import Markdown, display

reports = sorted(pathlib.Path('runs/sweep').glob('*/report.md'))
display(Markdown(reports[-1].read_text()))

## Triage loop (Phase 2)

For each non-pass item in the report:

1. Diagnose — learning rate, schedule, tier budget (`core/tiers.py` overrides), or a real bug.
2. Fix in source, then re-run just that item:
   ```
   python main.py sweep --check --training_tier M --device cuda --models <name> --skip-compositions
   ```
3. Threshold changes go in `core/evalspec.py` **with a justification comment**.
4. Commit when green; note "what it took" for the docs chapter.